<a href="https://colab.research.google.com/github/TinaKgn/tourism_data_project/blob/absa-llm-validation/notebooks/users/kristina/exploratory/01_ABSA_NLI_Aspect_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ABSA Step 1: NLI-Based Aspect Presence Detection

**⚠️ Colab-Only Notebook** - This notebook is designed to run on Google Colab with GPU acceleration (T4 recommended).

**Purpose**: Detect which evaluative aspects are present in tourism reviews using DeBERTa NLI model with keyword validation.

**Output**: Parquet file containing nested dict structure with aspect presence flags only: `{aspect_name: {"present": bool}, ...}`

**Note**: This is Step 1 of a two-step ABSA pipeline. Sentiment classification is performed in the separate `02_ABSA_LLM_Sentiment_Evaluation.ipynb` notebook.

## Aspect Taxonomy

The following evaluative aspects are extracted and analyzed:

In [1]:
EVALUATIVE_ASPECTS = {
    # PRODUCT QUALITY
    "food quality": "quality and taste of food and meals in cafes and restaurants",
    "drink quality": "quality and taste of beverages in cafes and restaurants",
    "accommodation quality": "comfort, quality, and amenities of hotel accommodation",
    "tours": "quality of guided tours, tourism experiences, and tour guides",

    # SERVICE
    "service quality": "speed, efficiency, and attentiveness of service",
    "staff friendliness": "politeness, helpfulness, and professionalism of staff",

    # EXPERIENCE
    "wait time": "time spent waiting for service, tables, or seating",
    "crowding": "overcrowding in restaurants, hotel dining areas, breakfast rooms, lobbies, elevators",
    "availability": "difficulty booking hotel reservations or getting a table at restaurants",
    "noise level": "amount of noise and sound disturbance",
    "room size": "spaciousness and dimensions of rooms and private spaces",

    # ENVIRONMENT
    "atmosphere": "ambiance and overall mood of the establishment",
    "cleanliness": "hygiene and tidiness of rooms and facilities",
    "location": "proximity and accessibility of the establishment to attractions and transportation",

    # VALUE
    "value for money": "price fairness relative to quality received"
}

## Setup and Dependencies

In [ ]:
import os
import sys
import pickle
import hashlib
import shutil
from typing import List, Dict, Optional
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# =============================================================================
# COLAB SETUP: Mount Google Drive
# =============================================================================

print("🌐 Running on Google Colab")
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = '/content/drive/MyDrive/tourism_data_project'
if not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive mount unavailable at /content/drive/MyDrive')
if not os.path.isdir(DRIVE_BASE):
    raise FileNotFoundError(f'Expected project folder not found in Drive: {DRIVE_BASE}')
print(f"✅ Drive path verified: {DRIVE_BASE}")

In [3]:
# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Enable GPU in Runtime > Change runtime type")

Using device: cuda
GPU: Tesla T4


## Load and Prepare Data

In [ ]:
# Load TripAdvisor data from Google Drive
data_path = os.path.join(
    DRIVE_BASE,
    "data/silver/tripadvisor/staged_primary"
    "/tripadvisor_nyc_2015_2023_date_filtered_staged.parquet"
)
print(f"📂 Loading data from: {data_path}")

df = pd.read_parquet(data_path)
print(f"✅ Data loaded: {len(df)} rows")

# Filter to Pre-COVID (2015-2019) and Post-Recovery (2022-2023) periods
# Exclude 2020-2021 entirely
df = df[df['rvw_year_pr'].isin([2015, 2016, 2017, 2018, 2019, 2022, 2023])].copy()
print(f"✅ Filtered to 2015-2019 + 2022-2023: {len(df)} rows")

# Temporal distribution
year_dist = df['rvw_year_pr'].value_counts().sort_index()
print("\n📅 Year Distribution:")
for year, count in year_dist.items():
    print(f"  {year}: {count:,} reviews")

In [5]:
# Create a deterministic stratified sample of 5000 TripAdvisor reviews
# Stratify by VADER tercile only with temporal balancing across 2015-2019 and 2022-2023 periods

sample_target = 5000
RANDOM_STATE = 42

# Ensure required columns exist in the dataframe
required_cols = ['rvw_vader_pr', 'rvw_text_pr', 'rvw_year_pr', 'rvw_month_pr']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f'Missing required columns for sampling: {missing}')

# Add derived features
df = df.copy()

# Create tercile bins for vader only
df['vader_bin'] = pd.qcut(df['rvw_vader_pr'].rank(method='first'), q=3, labels=['low','mid','high'])

# Year-month as str for grouping
df['rvw_month_pr'] = df['rvw_month_pr'].fillna(0).astype(int)
df['month_str'] = df['rvw_month_pr'].astype(str).str.zfill(2)
df['year_month'] = df['rvw_year_pr'].astype(str) + '-' + df['month_str']

# Primary stratification: vader_bin only (3 strata)
df['primary_strata'] = df['vader_bin'].astype(str)

# Temporal priority: ensure balanced month representation
import numpy as _np
_rng = _np.random.default_rng(RANDOM_STATE)

sample_idx = []

# Step 1: Sample proportionally from each primary stratum
primary_counts = df['primary_strata'].value_counts()
primary_proportions = primary_counts / len(df)

for stratum in primary_counts.index:
    stratum_df = df[df['primary_strata'] == stratum]

    # Allocate sample size proportionally
    stratum_target = int(_np.round(primary_proportions[stratum] * sample_target))

    if stratum_target == 0:
        continue

    # Within each primary stratum, sample uniformly across year-months
    year_month_groups = stratum_df.groupby('year_month')
    n_months = len(year_month_groups)

    if n_months == 0:
        continue

    # Allocate reviews per month within this stratum
    per_month = max(1, stratum_target // n_months)
    remainder = stratum_target % n_months

    for i, (ym, ym_group) in enumerate(year_month_groups):
        # Add 1 extra review to first 'remainder' months
        month_n = per_month + (1 if i < remainder else 0)

        available = ym_group.index.tolist()
        if len(available) <= month_n:
            sample_idx.extend(available)
        else:
            chosen = _rng.choice(available, size=month_n, replace=False).tolist()
            sample_idx.extend(chosen)

# Step 2: Adjust to exact target if needed
if len(sample_idx) < sample_target:
    remaining_needed = sample_target - len(sample_idx)
    pool = df.index.difference(sample_idx).tolist()
    if len(pool) > 0:
        extra = _rng.choice(pool, size=min(remaining_needed, len(pool)), replace=False).tolist()
        sample_idx.extend(extra)
elif len(sample_idx) > sample_target:
    sample_idx = sample_idx[:sample_target]

# Create sample dataframe preserving original indices
sample_df = df.loc[sample_idx].copy()
sample_df.reset_index(inplace=True)
sample_df.rename(columns={'index':'orig_index'}, inplace=True)

print(f"✅ Created sample with {len(sample_df)} reviews")
print(f"   Primary strata represented: {sample_df['primary_strata'].nunique()}/3")
print(f"   Year-months represented: {sample_df['year_month'].nunique()}/84")

# Temporal balance check
print("\n📅 Sample temporal distribution:")
year_sample_dist = sample_df['rvw_year_pr'].value_counts().sort_index()
for year, count in year_sample_dist.items():
    pct = 100 * count / len(sample_df)
    print(f"  {year}: {count:>4} reviews ({pct:>5.1f}%)")

# Stratum balance check
print("\n📊 Sample stratification balance (VADER terciles):")
strata_dist = sample_df['primary_strata'].value_counts().sort_index()
for stratum, count in strata_dist.items():
    pct = 100 * count / len(sample_df)
    print(f"  {stratum:.<20} {count:>4} ({pct:>5.1f}%)")

✅ Created sample with 5000 reviews
   Primary strata represented: 3/3
   Year-months represented: 73/84

📅 Sample temporal distribution:
  2015:  828 reviews ( 16.6%)
  2016:  828 reviews ( 16.6%)
  2017:  828 reviews ( 16.6%)
  2018:  828 reviews ( 16.6%)
  2019:  828 reviews ( 16.6%)
  2022:  795 reviews ( 15.9%)
  2023:   65 reviews (  1.3%)

📊 Sample stratification balance (VADER terciles):
  high................ 1667 ( 33.3%)
  low................. 1666 ( 33.3%)
  mid................. 1667 ( 33.3%)


## NLI-Based Aspect Detection Pipeline

In [6]:
class AspectPresenceDetector:
    """
    NLI-based aspect presence detection pipeline using DeBERTa.

    Features:
    - Batched inference
    - Disk caching to avoid recomputation
    - Memory-efficient chunked processing
    - Keyword-aware validation for crowding detection
    - Cache management utilities
    """

    def __init__(
        self,
        evaluative_aspects: Dict[str, str],
        nli_model_name: str = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
        entailment_threshold: float = 0.75,
        batch_cache_dir: str = "cache",
        device: Optional[str] = None
    ):
        # Store configuration
        self.evaluative_aspects = evaluative_aspects
        self.entailment_threshold = entailment_threshold

        # Decide device automatically if not specified
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        # Cache directories
        self.nli_cache_dir = os.path.join(batch_cache_dir, "nli")

        # Create cache directories
        os.makedirs(self.nli_cache_dir, exist_ok=True)

        # Load NLI model (aspect presence)
        print("Loading NLI model...")
        self.nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
        self.nli_model = AutoModelForSequenceClassification.from_pretrained(
            nli_model_name
        ).to(self.device)
        self.nli_model.eval()
        print("✅ NLI model loaded successfully!")

    # Helper methods

    def _aspect_hypothesis(self, aspect: str) -> str:
        """Convert an aspect into a hypothesis sentence suitable for NLI-based aspect detection."""
        return (
            f"The review mentions or discusses "
            f"{self.evaluative_aspects[aspect]}."
        )

    def _hash_batch(self, items: List[str]) -> str:
        """Create a deterministic hash for a list of strings."""
        joined = "||" .join(items)
        return hashlib.md5(joined.encode("utf-8")).hexdigest()

    def _load_cache(self, path: str):
        """Load cached object from disk if it exists."""
        if os.path.exists(path):
            try:
                with open(path, "rb") as f:
                    return pickle.load(f)
            except (pickle.PickleError, EOFError):
                print(f"Warning: Cache file corrupted, removing: {path}")
                os.remove(path)
        return None

    def _save_cache(self, path: str, obj):
        """Save object to disk cache."""
        with open(path, "wb") as f:
            pickle.dump(obj, f)

    # Cache management methods

    def get_cache_size(self) -> Dict[str, float]:
        """Get the total size of the NLI cache directory in MB."""
        def get_dir_size(path):
            total = 0
            if os.path.exists(path):
                for dirpath, dirnames, filenames in os.walk(path):
                    for f in filenames:
                        fp = os.path.join(dirpath, f)
                        if os.path.exists(fp):
                            total += os.path.getsize(fp)
            return total / (1024 * 1024)  # Convert to MB

        return {"total_cache_mb": get_dir_size(self.nli_cache_dir)}

    def clear_cache(self, confirm: bool = True) -> Dict[str, str]:
        """Clear NLI cache files."""
        cache_sizes = self.get_cache_size()

        if confirm:
            print(f"\nCurrent cache size: {cache_sizes['total_cache_mb']:.2f} MB")
            response = input(f"\nAre you sure you want to clear NLI cache? (yes/no): ")
            if response.lower() != "yes":
                return {"status": "cancelled", "message": "Cache clearing cancelled by user"}

        if os.path.exists(self.nli_cache_dir):
            shutil.rmtree(self.nli_cache_dir)
            os.makedirs(self.nli_cache_dir, exist_ok=True)
            return {"status": "success", "message": f"Cleared {cache_sizes['total_cache_mb']:.2f} MB"}
        else:
            return {"status": "success", "message": "No cache to clear"}

    # Validation methods for problematic aspects

    def _validate_crowding_detection(self, review_text: str, nli_score: float) -> bool:
        """Validate crowding detection using keyword presence + NLI score."""
        strong_keywords = [
            'crowded', 'crowding',
            'packed', 'cramped', 'crammed',
            'congested', 'too many people', 'many people',
            'overcrowd', 'overcrowded','mobbed','jammed','queue'
        ]
        weak_keywords = ['full','busy']

        review_lower = review_text.lower()

        # Check for strong keywords
        has_strong_keyword = any(keyword in review_lower for keyword in strong_keywords)
        if has_strong_keyword:
            return True

        # Check for weak keywords
        has_weak_keyword = any(keyword in review_lower for keyword in weak_keywords)
        if has_weak_keyword:
            result = nli_score >= 0.7
            return result

        # No crowding keywords at all
        result = nli_score >= 0.85
        return result

    # Batched NLI aspect presence detection

    def _batch_detect_aspect_presence(
        self,
        reviews: List[str],
        batch_size: int
    ):
        """Detect which evaluative aspects are present in each review using batched NLI inference."""
        # Prepare output dictionary
        results = {i: {} for i in range(len(reviews))}

        # Build all (review, hypothesis) pairs
        pairs = []
        meta = []

        for review_idx, review in enumerate(reviews):
            for aspect in self.evaluative_aspects:
                hypothesis = self._aspect_hypothesis(aspect)
                pairs.append((review, hypothesis))
                meta.append((review_idx, aspect))

        # Process in batches
        for i in tqdm(range(0, len(pairs), batch_size), desc="Detecting aspects"):
            batch_pairs = pairs[i:i + batch_size]
            batch_meta = meta[i:i + batch_size]

            # Build cache key
            cache_key = self._hash_batch(
                [f"{r}::{h}" for r, h in batch_pairs]
            )
            cache_path = os.path.join(self.nli_cache_dir, f"{cache_key}.pkl")

            cached_scores = self._load_cache(cache_path)

            if cached_scores is None:
                premises, hypotheses = zip(*batch_pairs)

                inputs = self.nli_tokenizer(
                    list(premises),
                    list(hypotheses),
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=512
                ).to(self.device)

                with torch.no_grad():
                    logits = self.nli_model(**inputs).logits

                probs = torch.softmax(logits, dim=-1)

                # Handle different label formats (case-insensitive)
                label2id = {k.lower(): v for k, v in self.nli_model.config.label2id.items()}
                entailment_idx = label2id.get("entailment")

                if entailment_idx is None:
                    entailment_idx = 2
                    print("Warning: 'entailment' label not found, using index 2 as fallback")

                # Save entailment probabilities only
                batch_scores = probs[:, entailment_idx].cpu().tolist()
                self._save_cache(cache_path, batch_scores)
            else:
                batch_scores = cached_scores

            # Assign results with validation
            for score, (review_idx, aspect) in zip(batch_scores, batch_meta):
                # Special handling for problematic aspects
                if aspect == "crowding":
                    detected = self._validate_crowding_detection(
                        reviews[review_idx], score
                    )
                else:
                    detected = score >= self.entailment_threshold

                results[review_idx][aspect] = {"present": detected}

        return results

    # Process all reviews

    def process_reviews(
        self,
        reviews: List[str],
        batch_size: int = 64,
        chunk_size: int = 1000
    ):
        """Process a list of reviews and return aspect presence annotations."""
        # Input validation
        if not reviews:
            return []

        if batch_size < 1:
            raise ValueError("batch_size must be positive")

        # Filter empty reviews
        valid_indices = [i for i, r in enumerate(reviews) if r and r.strip()]
        valid_reviews = [reviews[i] for i in valid_indices]

        if not valid_reviews:
            return []

        print(f"Processing {len(valid_reviews)} valid reviews...")

        all_results = []

        # Process in chunks for memory efficiency
        for chunk_start in range(0, len(valid_reviews), chunk_size):
            chunk_end = min(chunk_start + chunk_size, len(valid_reviews))
            chunk_reviews = valid_reviews[chunk_start:chunk_end]

            print(f"\nProcessing chunk {chunk_start//chunk_size + 1} ({chunk_start}-{chunk_end})...")

            # Initialize result structure
            chunk_results = [{"text": r, "aspects": {}} for r in chunk_reviews]

            # Aspect presence detection
            presence = self._batch_detect_aspect_presence(
                chunk_reviews, batch_size
            )

            # Attach presence to results
            for i in range(len(chunk_reviews)):
                chunk_results[i]["aspects"] = presence[i]

            all_results.extend(chunk_results)

            # Clear GPU cache if using CUDA
            if self.device == "cuda":
                torch.cuda.empty_cache()

        return all_results

## Cache Directory Setup

In [ ]:
# Cache directory on Google Drive (persists across Colab sessions)
cache_dir = os.path.join(DRIVE_BASE, "data/reference/cache")
os.makedirs(cache_dir, exist_ok=True)
print(f"✅ Cache directory ready: {cache_dir}")

## Run NLI Aspect Detection

In [ ]:
# Initialize detector and run on sampled reviews
detector = AspectPresenceDetector(
    evaluative_aspects=EVALUATIVE_ASPECTS,
    entailment_threshold=0.75,
    batch_cache_dir=cache_dir
)

# Prepare sample reviews (preserve order)
sample_reviews = sample_df['rvw_text_pr'].astype(str).tolist()
print(f'Running NLI aspect detection on {len(sample_reviews)} sampled reviews...')

# Process the sample
results = detector.process_reviews(
    sample_reviews,
    batch_size=8,
    chunk_size=200
)

print(f'✅ NLI aspect detection produced {len(results)} results')

## Save NLI Results

In [ ]:
# Convert results to DataFrame with nested aspects
results_df = pd.DataFrame(results)

# Define primary schema columns
primary_cols = [
    'rvw_id_pr', 'usr_id_pr', 'lst_id_pr', 'rvw_text_pr', 'rvw_text_flags_pr',
    'rvw_vader_pr', 'rvw_year_pr', 'rvw_month_pr', 'lst_lat_pr', 'lst_lon_pr', 'lst_metro_code_pr',
    'is_accommodation_pr', 'is_restaurant_pr', 'is_nightlife_pr', 'is_entertainment_pr', 'is_tours_pr', 'is_events_pr'
]

# Start with sample_df and ensure all primary columns exist
staged = sample_df.copy()
for col in primary_cols:
    if col not in staged.columns:
        staged[col] = None

# Attach NLI aspects dict as a secondary enrichment column
if 'aspects' in results_df.columns:
    staged['nli_aspects'] = results_df['aspects'].astype(object).values
else:
    staged['nli_aspects'] = [{} for _ in range(len(staged))]

# Assemble output columns: original index + primary schema + NLI aspects
output_cols = ['orig_index'] + primary_cols + ['nli_aspects']
staged_output = staged.reindex(columns=output_cols)

# Save to Google Drive
out_dir = os.path.join(DRIVE_BASE, 'data', 'silver', 'tripadvisor', 'staged_primary')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'tripadvisor_nyc_absa_nli_aspects_5000_staged.parquet')
staged_output.to_parquet(out_path, index=False)
print(f'✅ Saved NLI aspect detection results to: {out_path}')

## Verify Output

In [10]:
# Load and verify the saved results
df_nli = pd.read_parquet(out_path)
print(f"✅ Loaded NLI results: {len(df_nli)} rows")
print(f"Columns: {df_nli.columns.tolist()}\n")

# Sample output inspection
print("Sample NLI aspects structure (first review):")
print(df_nli['nli_aspects'].iloc[0])

# Count aspects with detection
def count_detected_aspects(aspects_dict):
    if not isinstance(aspects_dict, dict):
        return 0
    return sum(1 for v in aspects_dict.values() if isinstance(v, dict) and v.get('present', False))

df_nli['aspect_count'] = df_nli['nli_aspects'].apply(count_detected_aspects)

print(f"\n📊 Aspect Detection Summary:")
print(f"  Reviews with ≥1 aspect detected: {(df_nli['aspect_count'] > 0).sum()} / {len(df_nli)}")
print(f"  Average aspects per review: {df_nli['aspect_count'].mean():.2f}")
print(f"  Max aspects in single review: {df_nli['aspect_count'].max()}")

✅ Loaded NLI results: 5000 rows
Columns: ['orig_index', 'rvw_id_pr', 'usr_id_pr', 'lst_id_pr', 'rvw_text_pr', 'rvw_text_flags_pr', 'rvw_vader_pr', 'rvw_year_pr', 'rvw_month_pr', 'lst_lat_pr', 'lst_lon_pr', 'lst_metro_code_pr', 'is_accommodation_pr', 'is_restaurant_pr', 'is_nightlife_pr', 'is_entertainment_pr', 'is_tours_pr', 'is_events_pr', 'nli_aspects']

Sample NLI aspects structure (first review):
{'accommodation quality': {'present': True}, 'atmosphere': {'present': True}, 'availability': {'present': True}, 'cleanliness': {'present': True}, 'crowding': {'present': False}, 'drink quality': {'present': False}, 'food quality': {'present': False}, 'location': {'present': False}, 'noise level': {'present': False}, 'room size': {'present': True}, 'service quality': {'present': True}, 'staff friendliness': {'present': True}, 'tours': {'present': False}, 'value for money': {'present': True}, 'wait time': {'present': True}}

📊 Aspect Detection Summary:
  Reviews with ≥1 aspect detected: 499

## Cache Cleanup (Optional)

In [ ]:
# Set to True to delete the NLI cache after the pipeline completes.
# The cache stores intermediate inference results and allows interrupted
# runs to resume without recomputing. Keep it if you may re-run this notebook.
CLEAR_CACHE_ON_COMPLETION = False

if CLEAR_CACHE_ON_COMPLETION:
    print("\n🧹 Cleaning up cache...")
    detector.clear_cache(confirm=False)
    print("✅ Cache cleared successfully")
else:
    cache_size = detector.get_cache_size()
    print(f"💾 Cache preserved: {cache_size['total_cache_mb']:.2f} MB")
    print("   Set CLEAR_CACHE_ON_COMPLETION = True to delete it.")